## 26 — Citing Papers Data Collection

For each award-winning paper we fetch the papers that **cite it** (forward citations).
This is the reverse direction of notebook 24 (`referenced_works` = backward citations).

**OpenAlex mechanism:**  
`GET /works?filter=cites:W<id>&per-page=200&cursor=*`  
This returns all works that cite a given paper, paginated via cursor.

**Workflow:**
1. Load `huang_matched_openalex.csv` → unique award paper OpenAlex IDs
2. For each award paper: paginate through all citing works, collect their IDs
3. Build edge table: `award_paper_id → citing_paper_id`
4. Deduplicate citing IDs, batch-fetch full metadata (same `extract_paper_attrs` as nb-24)
5. Save `citing_papers.csv` and `award_to_citing_edges.csv`

**Output files:**
- `citing_papers.csv` — one row per unique citing paper, same attributes as award papers
- `award_to_citing_edges.csv` — explicit mapping: which award paper is cited by which paper

**Space / performance notes:**
- `counts_by_year` stored as compact JSON string (same as nb-24)
- `citing_award_papers` stored as `|`-separated ID string (not a list)
- Checkpoint every 25 award papers (citing lists can be large → save often)
- Batch size 200 for metadata fetch (max OpenAlex allows)

In [1]:
import pandas as pd
import requests
import json
import time
from pathlib import Path

MAILTO   = 'shaheryar.4822@student.uu.se'
API_KEY  = 'A08hCjeUoeVKA9toVsfCpF'
BASE_URL = 'https://api.openalex.org'

DATA = Path(r'B:\Semester 4 UU\thesis-best-paper-trajectories\data\matched')

MATCHED_PATH      = DATA / 'huang_matched_openalex.csv'
OUT_CITING        = DATA / 'citing_papers.csv'
OUT_EDGES         = DATA / 'award_to_citing_edges.csv'
CHECKPOINT_EDGES  = DATA / 'citing_edges_checkpoint.csv'
CHECKPOINT_PAPERS = DATA / 'citing_papers_checkpoint.csv'

# Select fields — identical to nb-24 to keep datasets consistent
SELECT_FIELDS = (
    'id,doi,title,publication_year,publication_date,type,'
    'cited_by_count,is_retracted,open_access,primary_location,'
    'best_oa_location,authorships,topics,counts_by_year'
)

def api_get(url, params=None):
    p = {'mailto': MAILTO, 'api_key': API_KEY}
    if params:
        p.update(params)
    for attempt in range(4):
        try:
            r = requests.get(url, params=p, timeout=20)
            if r.status_code == 200:
                return r.json()
            if r.status_code == 429:
                time.sleep(10 * (attempt + 1))
        except requests.RequestException:
            time.sleep(3)
    return None

print('Ready.')

Ready.


### Step 1 — Load award papers

In [2]:
df = pd.read_csv(MATCHED_PATH)
df = df[df['year'].between(2000, 2018)].copy()
df = df.dropna(subset=['openalex_id'])

award_papers = (
    df.drop_duplicates(subset='openalex_id')
    [['openalex_id', 'year', 'conference', 'paper_title']]
    .copy()
)
award_papers['openalex_id'] = award_papers['openalex_id'].str.strip()

print(f'Total award papers (2000-2018): {len(award_papers)}')
print(f'Year range: {award_papers["year"].min()} – {award_papers["year"].max()}')
print(award_papers.head(3))

Total award papers (2000-2018): 890
Year range: 2000 – 2018
                        openalex_id  year conference  \
0  https://openalex.org/W2788603415  2018       AAAI   
1  https://openalex.org/W2798397965  2018        ACL   
2  https://openalex.org/W2963033005  2018        ACL   

                                         paper_title  
0           Memory-Augmented Monte Carlo Tree Search  
1  Finding syntax in human encephalography with b...  
2  Learning to Ask Good Questions: Ranking Clarif...  


### Step 2 — Paginate citing works for each award paper

OpenAlex cursor pagination: start with `cursor=*`, then follow `meta.next_cursor` until `None`.
We only store IDs at this stage to keep memory low.

In [3]:
def fetch_citing_ids(award_oa_id):
    """Return list of (citing_paper_id,) for all papers citing award_oa_id."""
    short_id = award_oa_id.split('/')[-1]
    cursor   = '*'
    ids      = []

    while cursor:
        data = api_get(
            f'{BASE_URL}/works',
            params={
                'filter'  : f'cites:{short_id}',
                'select'  : 'id',
                'per-page': 200,
                'cursor'  : cursor,
            }
        )
        time.sleep(0.12)

        if not data:
            break

        results = data.get('results', [])
        ids.extend(r['id'] for r in results if r.get('id'))

        cursor = data.get('meta', {}).get('next_cursor')  # None when last page

    return ids


# Resume from checkpoint if available
if CHECKPOINT_EDGES.exists():
    edges_df = pd.read_csv(CHECKPOINT_EDGES)
    done_ids = set(edges_df['award_paper_id'].unique())
    edges    = edges_df.to_dict('records')
    print(f'Resuming — {len(done_ids)} award papers already processed, {len(edges):,} edges')
else:
    edges    = []
    done_ids = set()

to_process = award_papers[~award_papers['openalex_id'].isin(done_ids)]
print(f'Award papers left to process: {len(to_process)}')

for i, row in enumerate(to_process.itertuples(), 1):
    oa_id   = row.openalex_id
    cit_ids = fetch_citing_ids(oa_id)

    for cid in cit_ids:
        edges.append({
            'award_paper_id'   : oa_id,
            'award_year'       : row.year,
            'award_conference' : row.conference,
            'citing_paper_id'  : cid,
        })

    if i % 25 == 0:
        pd.DataFrame(edges).to_csv(CHECKPOINT_EDGES, index=False)
        print(f'  [{i}/{len(to_process)}] checkpoint — {len(edges):,} edges | last paper: {len(cit_ids)} citers')

edges_df = pd.DataFrame(edges)
edges_df.to_csv(CHECKPOINT_EDGES, index=False)

print(f'\nTotal citing edges      : {len(edges_df):,}')
print(f'Unique citing paper IDs : {edges_df["citing_paper_id"].nunique():,}')

Award papers left to process: 890
  [25/890] checkpoint — 2,806 edges | last paper: 103 citers
  [50/890] checkpoint — 5,800 edges | last paper: 70 citers
  [75/890] checkpoint — 9,461 edges | last paper: 21 citers
  [100/890] checkpoint — 13,041 edges | last paper: 105 citers
  [125/890] checkpoint — 60,382 edges | last paper: 60 citers
  [150/890] checkpoint — 92,606 edges | last paper: 0 citers
  [175/890] checkpoint — 94,561 edges | last paper: 65 citers
  [200/890] checkpoint — 316,459 edges | last paper: 1309 citers
  [225/890] checkpoint — 319,809 edges | last paper: 9 citers
  [250/890] checkpoint — 322,473 edges | last paper: 44 citers
  [275/890] checkpoint — 327,237 edges | last paper: 28 citers
  [300/890] checkpoint — 329,658 edges | last paper: 75 citers
  [325/890] checkpoint — 333,960 edges | last paper: 38 citers
  [350/890] checkpoint — 337,809 edges | last paper: 45 citers
  [375/890] checkpoint — 351,506 edges | last paper: 66 citers
  [400/890] checkpoint — 354,904

### Step 3 — Build `cites_n_award_papers` lookup & deduplicate

In [4]:
# For each citing paper: list of award papers it cites + count
cites_map = (
    edges_df.groupby('citing_paper_id')['award_paper_id']
    .apply(lambda x: '|'.join(sorted(x.unique())))
    .reset_index()
    .rename(columns={'award_paper_id': 'cites_award_papers'})
)

cites_count = (
    edges_df.groupby('citing_paper_id')['award_paper_id']
    .nunique()
    .reset_index()
    .rename(columns={'award_paper_id': 'cites_n_award_papers'})
)

citing_meta     = cites_map.merge(cites_count, on='citing_paper_id')
unique_cite_ids = citing_meta['citing_paper_id'].tolist()

print(f'Unique citing papers to enrich : {len(unique_cite_ids):,}')
print(f'Cite >= 2 award papers          : {(citing_meta["cites_n_award_papers"] >= 2).sum():,}')
print(f'Cite >= 5 award papers          : {(citing_meta["cites_n_award_papers"] >= 5).sum():,}')

Unique citing papers to enrich : 438,807
Cite >= 2 award papers          : 61,123
Cite >= 5 award papers          : 331


### Step 4 — Batch-fetch citing paper metadata

Batch size 200 (OpenAlex OR filter limit).  
Same `extract_paper_attrs` as notebook 24 for dataset consistency.

In [11]:
def extract_paper_attrs(w):
    primary_loc = w.get('primary_location') or {}
    source      = primary_loc.get('source') or {}

    authorships = w.get('authorships') or []

    # Guard every .get() with 'or empty-string' — OpenAlex can return null for any field
    author_ids = '|'.join([
        a['author']['id']
        for a in authorships
        if (a.get('author') or {}).get('id')
    ])

    author_names = '|'.join([
        (a.get('author') or {}).get('display_name') or ''
        for a in authorships
        if a.get('author')
    ])

    author_positions = '|'.join([
        a.get('author_position') or ''
        for a in authorships
    ])

    first_insts = []
    if authorships:
        first_insts = [
            (i.get('display_name') or '')
            for i in (authorships[0].get('institutions') or [])
        ]

    topics    = w.get('topics') or []
    top_topic = (topics[0].get('display_name') or '') if topics else ''
    top_field = ((topics[0].get('field') or {}).get('display_name') or '') if topics else ''

    return {
        'openalex_id'              : w.get('id') or '',
        'doi'                      : w.get('doi') or '',
        'title'                    : w.get('title') or '',
        'publication_year'         : w.get('publication_year'),
        'publication_date'         : w.get('publication_date') or '',
        'type'                     : w.get('type') or '',
        'cited_by_count'           : w.get('cited_by_count') or 0,
        'is_retracted'             : w.get('is_retracted') or False,
        'is_oa'                    : (w.get('open_access') or {}).get('is_oa') or False,
        'source_id'                : source.get('id') or '',
        'source_name'              : source.get('display_name') or '',
        'source_type'              : source.get('type') or '',
        'source_issn'              : '|'.join(source.get('issn') or []),
        'author_ids'               : author_ids,
        'author_names'             : author_names,
        'author_positions'         : author_positions,
        'author_count'             : len(authorships),
        'first_author_institution' : '|'.join(first_insts),
        'top_topic'                : top_topic,
        'top_field'                : top_field,
        'counts_by_year'           : json.dumps(w.get('counts_by_year') or []),
    }


In [14]:
def fetch_works_batch(id_list):
    ids_str = '|'.join([i.split('/')[-1] for i in id_list])
    data = api_get(
        f'{BASE_URL}/works',
        params={
            'filter'  : f'openalex_id:{ids_str}',
            'per-page': 50,
            'select'  : SELECT_FIELDS,
        }
    )
    if data:
        return data.get('results', [])
    return []


# Resume from checkpoint (auto-deletes if empty/broken)
if CHECKPOINT_PAPERS.exists():
    probe = pd.read_csv(CHECKPOINT_PAPERS)
    if len(probe) == 0 or probe['openalex_id'].isna().all():
        CHECKPOINT_PAPERS.unlink()
        print('Deleted empty checkpoint, starting fresh.')

if CHECKPOINT_PAPERS.exists():
    existing    = pd.read_csv(CHECKPOINT_PAPERS)
    fetched_ids = set(existing['openalex_id'].dropna().tolist())
    all_papers  = existing.to_dict('records')
    print(f'Resuming — {len(fetched_ids):,} citing papers already fetched')
else:
    fetched_ids = set()
    all_papers  = []

remaining = [i for i in unique_cite_ids if i not in fetched_ids]
print(f'Citing papers left to fetch: {len(remaining):,}')

BATCH   = 50
batches = [remaining[i:i+BATCH] for i in range(0, len(remaining), BATCH)]
print(f'Total batches: {len(batches):,}')

if batches:
    test = fetch_works_batch(batches[0])
    print(f'Batch 1 sanity check: {len(test)} results returned (expected ~50)')
    assert len(test) > 0, 'STOP: batch 1 returned 0 — check API key / URL'

for b_idx, batch in enumerate(batches, 1):
    results = fetch_works_batch(batch)
    time.sleep(0.15)

    for w in results:
        all_papers.append(extract_paper_attrs(w))

    if b_idx % 200 == 0:
        pd.DataFrame(all_papers).to_csv(CHECKPOINT_PAPERS, index=False)
        pct = b_idx / len(batches) * 100
        print(f'  [batch {b_idx}/{len(batches)} | {pct:.1f}%] {len(all_papers):,} papers fetched')

citing_papers_df = pd.DataFrame(all_papers)
citing_papers_df.to_csv(CHECKPOINT_PAPERS, index=False)
print(f'\nTotal citing papers fetched: {len(citing_papers_df):,}')

C:\Users\sla99\AppData\Local\Temp\ipykernel_33336\3403138487.py:18: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  probe = pd.read_csv(CHECKPOINT_PAPERS)
C:\Users\sla99\AppData\Local\Temp\ipykernel_33336\3403138487.py:24: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  existing    = pd.read_csv(CHECKPOINT_PAPERS)


Resuming — 300,000 citing papers already fetched
Citing papers left to fetch: 138,807
Total batches: 2,777
Batch 1 sanity check: 50 results returned (expected ~50)
  [batch 200/2777 | 7.2%] 310,000 papers fetched
  [batch 400/2777 | 14.4%] 320,000 papers fetched
  [batch 600/2777 | 21.6%] 330,000 papers fetched
  [batch 800/2777 | 28.8%] 340,000 papers fetched
  [batch 1000/2777 | 36.0%] 350,000 papers fetched
  [batch 1200/2777 | 43.2%] 360,000 papers fetched
  [batch 1400/2777 | 50.4%] 370,000 papers fetched
  [batch 1600/2777 | 57.6%] 380,000 papers fetched
  [batch 1800/2777 | 64.8%] 390,000 papers fetched
  [batch 2000/2777 | 72.0%] 400,000 papers fetched
  [batch 2200/2777 | 79.2%] 410,000 papers fetched
  [batch 2400/2777 | 86.4%] 420,000 papers fetched
  [batch 2600/2777 | 93.6%] 430,000 papers fetched

Total citing papers fetched: 438,807


### Step 5 — Merge metadata & save final outputs

In [15]:
final_citing = citing_papers_df.merge(
    citing_meta, left_on='openalex_id', right_on='citing_paper_id', how='left'
)
final_citing = final_citing.drop(columns=['citing_paper_id'], errors='ignore')

# Compact dtypes to save disk space
for col in ['cited_by_count', 'author_count', 'cites_n_award_papers']:
    if col in final_citing.columns:
        final_citing[col] = pd.to_numeric(final_citing[col], errors='coerce').astype('Int32')

final_citing['publication_year'] = pd.to_numeric(
    final_citing['publication_year'], errors='coerce'
).astype('Int16')

# Save
final_citing.to_csv(OUT_CITING, index=False)
edges_df.to_csv(OUT_EDGES, index=False)

print('=' * 60)
print(f'citing_papers.csv         → {len(final_citing):,} rows')
print(f'award_to_citing_edges.csv → {len(edges_df):,} rows')
print(f'\ncites_n_award_papers distribution (top 10):')
print(final_citing['cites_n_award_papers'].value_counts().sort_index().head(10))
print(f'\nMissing cites_award_papers: {final_citing["cites_award_papers"].isna().sum()}')
print(f'\nYear range of citing papers:')
print(f'  min: {final_citing["publication_year"].min()}')
print(f'  max: {final_citing["publication_year"].max()}')
print(f'  median: {final_citing["publication_year"].median()}')
print(f'\nSample rows:')
print(
    final_citing[['openalex_id', 'title', 'publication_year',
                   'cited_by_count', 'cites_n_award_papers']]
    .head(3).to_string()
)

citing_papers.csv         → 438,807 rows
award_to_citing_edges.csv → 507,718 rows

cites_n_award_papers distribution (top 10):
cites_n_award_papers
1     377684
2      55534
3       4658
4        600
5        226
6         30
7         14
8          6
9          3
10         1
Name: count, dtype: Int64

Missing cites_award_papers: 0

Year range of citing papers:
  min: 1958
  max: 2026
  median: 2021.0

Sample rows:
                        openalex_id                                                                              title  publication_year  cited_by_count  cites_n_award_papers
0   https://openalex.org/W100367037  A Unified Framework for Multi-target Tracking and Collective Activity Recognition              2012             361                     1
1  https://openalex.org/W1006997171      Curiosity, Creativity, and Surprise as Analytic Tools: Grounded Theory Method              2014             112                     1
2   https://openalex.org/W100599422                    